# Rapport de performance

Ce rapport lit `data/06_performance/performance_metrics.csv`. Il ne réestime pas les poids et il ne rouvre pas les cours. Le tableau de bord est `reports/performance/dashboard.html`. L'analyse est `notebooks/07_analyse_performance.ipynb`.

Le rendement publié est mensuel, puis annualisé. Il n'y a pas de rendement journalier : le poids n'est pas refait chaque jour. Le rendement, la volatilité, le ratio de Sharpe et le drawdown maximal sont ceux de la phase 5. Le calcul s'arrête s'il ne les retrouve pas. Le ratio de Sortino et l'écart de suivi par rapport au fonds iShares sont ajoutés ici. La cible écrite de 8 % est ex ante. Elle ne choisit pas les poids. Les écarts ci-dessous sont mesurés après coup.

Un ratio de Sortino vide signifie qu'aucun mois n'est passé sous le SARON. Un ratio d'information vide signifie que l'écart de suivi est nul.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.performance.build_performance import (
    ACTIVE_RETURNS_FILE,
    DRAWDOWN_EPISODES_FILE,
    METRICS_OUTPUT,
)

LABELS = {
    "equal_weight": "Équipondéré",
    "minimum_volatility": "Volatilité minimale",
    "maximum_sharpe": "Sharpe maximal",
}


def french(value, digits):
    return f"{float(value):.{digits}f}".replace(".", ",")


def percent(value):
    return french(float(value) * 100, 1) + " %"


def ratio(value):
    if pd.isna(value):
        return "non publié"
    return french(value, 2)


metrics = pd.read_csv(METRICS_OUTPUT)
if not set(LABELS) <= set(metrics["strategy"]):
    raise RuntimeError("The performance file does not cover the three strategies.")
start = metrics["window_start"].iloc[0]
end = metrics["window_end"].iloc[0]
months = int(metrics["months"].iloc[0])
print(f"Période {start} – {end}, {months} mois.")
print(
    "Ces mesures décrivent le chemin des poids décidés chaque trimestre avec les soixante mois précédents, "
    "puis laissés dériver avec le marché. "
    "Chaque décision n'utilise que les rendements déjà observés."
)
print(f"La référence de l'écart de suivi est {metrics['reference'].iloc[0]}.")
print()
for strategy, label in LABELS.items():
    row = metrics.loc[metrics["strategy"].eq(strategy)].iloc[0]
    print(label)
    print(
        f"Rendement annualisé {percent(row['annualized_return'])}. "
        f"Volatilité {percent(row['annualized_volatility'])}."
    )
    print(f"Sharpe {ratio(row['sharpe_ratio'])}. Sortino {ratio(row['sortino_ratio'])}.")
    print(f"Drawdown maximal {percent(row['maximum_drawdown'])}.")
    print(f"Mois sous le SARON : {int(row['months_below_risk_free'])}.")
    print(
        f"Écart de suivi {percent(row['tracking_error'])}. "
        f"Ratio d'information {ratio(row['information_ratio'])}."
    )
    print()
print(
    "L'écart de suivi compare chaque portefeuille au rendement en francs "
    "de la valeur liquidative publiée par iShares."
)
print()
active = pd.read_csv(ACTIVE_RETURNS_FILE)
episodes = pd.read_csv(DRAWDOWN_EPISODES_FILE)
print("Écart mensuel avec le fonds")
for strategy, label in LABELS.items():
    book = active.loc[active["strategy"].eq(strategy)]
    ahead = int(book["active_return"].gt(0).sum())
    worst = book.loc[book["active_return"].idxmin()]
    episode = episodes.loc[episodes["strategy"].eq(strategy)].iloc[0]
    recovered = episode["recovered_month"]
    if pd.isna(recovered) or str(recovered).strip() == "":
        ending = "pas repris à la fin de la période."
    else:
        ending = f"reprise en {recovered}."
    print(
        f"{label} : {ahead} mois sur {len(book)} devant le fonds. "
        f"Plus mauvais mois {worst['month']} à {percent(worst['active_return'])}."
    )
    print(
        f"Drawdown : plus haut {episode['peak_month']}, "
        f"creux {episode['trough_month']} à {percent(episode['maximum_drawdown'])}, "
        f"{ending}"
    )


Période 2016-11 – 2026-10, 120 mois.
Ces mesures décrivent le chemin des poids décidés chaque trimestre avec les soixante mois précédents, puis laissés dériver avec le marché. Chaque décision n'utilise que les rendements déjà observés.
La référence de l'écart de suivi est iShares MSCI Global Sustainable Development Goals ETF.

Équipondéré
Rendement annualisé 8,0 %. Volatilité 12,3 %.
Sharpe 0,69. Sortino 1,09.
Drawdown maximal -22,9 %.
Mois sous le SARON : 49.
Écart de suivi 7,8 %. Ratio d'information 0,21.

Volatilité minimale
Rendement annualisé 6,3 %. Volatilité 11,8 %.
Sharpe 0,59. Sortino 0,90.
Drawdown maximal -23,1 %.
Mois sous le SARON : 50.
Écart de suivi 8,3 %. Ratio d'information 0,00.

Sharpe maximal
Rendement annualisé 6,3 %. Volatilité 12,1 %.
Sharpe 0,57. Sortino 0,84.
Drawdown maximal -27,4 %.
Mois sous le SARON : 47.
Écart de suivi 8,4 %. Ratio d'information 0,00.

L'écart de suivi compare chaque portefeuille au rendement en francs de la valeur liquidative publiée par 